In [83]:
!pip install kaggle

# Importamos las paqueterias necesarias

In [84]:
import pandas as pd
import re

# Importamos los datos mediante de la pagina de kaggle

In [85]:

## Primero tuvimos que crear un token en la pagina de kaggle para despues usarlo
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_0f4dfd11c4b106ceed1929b0ea36924a"

import kaggle
DEST = "./nba_data"
os.makedirs(DEST, exist_ok=True)

kaggle.api.authenticate()
kaggle.api.dataset_download_files("nathanlauga/nba-games", path=DEST, unzip=True)

df_games = pd.read_csv(f"{DEST}/games.csv")
df_games_details = pd.read_csv(f"{DEST}/games_details.csv")
df_players = pd.read_csv(f"{DEST}/players.csv")
df_ranking = pd.read_csv(f"{DEST}/ranking.csv")
df_teams = pd.read_csv(f"{DEST}/teams.csv")

print("Todo cargado")

Dataset URL: https://www.kaggle.com/datasets/nathanlauga/nba-games


C:\Users\chris\AppData\Local\Temp\ipykernel_15852\2538978692.py:14: DtypeWarning: Columns (0: NICKNAME) have mixed types. Specify dtype option on import or set low_memory=False.
  df_games_details = pd.read_csv(f"{DEST}/games_details.csv")


Todo cargado


## Análisis simple de las bases de datos

In [86]:
print("Información de df_games")
df_games.info()
print("")
print("-------------------------------")
print("")
print("Información de df_games_details")
df_games_details.info()
print("")
print("-------------------------------")
print("")
print("Información de df_players")
df_players.info()
print("")
print("-------------------------------")
print("")
print("Información de df_ranking")
df_ranking.info()
print("")
print("-------------------------------")
print("")
print("Información de df_teams")
df_teams.info()

Información de df_games
<class 'pandas.DataFrame'>
RangeIndex: 26651 entries, 0 to 26650
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   GAME_DATE_EST     26651 non-null  str    
 1   GAME_ID           26651 non-null  int64  
 2   GAME_STATUS_TEXT  26651 non-null  str    
 3   HOME_TEAM_ID      26651 non-null  int64  
 4   VISITOR_TEAM_ID   26651 non-null  int64  
 5   SEASON            26651 non-null  int64  
 6   TEAM_ID_home      26651 non-null  int64  
 7   PTS_home          26552 non-null  float64
 8   FG_PCT_home       26552 non-null  float64
 9   FT_PCT_home       26552 non-null  float64
 10  FG3_PCT_home      26552 non-null  float64
 11  AST_home          26552 non-null  float64
 12  REB_home          26552 non-null  float64
 13  TEAM_ID_away      26651 non-null  int64  
 14  PTS_away          26552 non-null  float64
 15  FG_PCT_away       26552 non-null  float64
 16  FT_PCT_away       26552 non

In [87]:
print("Información de df_games")
print(df_games.describe())
print("")
print("-------------------------------")
print("")
print("Información de df_games_details")
print(df_games_details.describe())
print("")
print("-------------------------------")
print("")
print("Información de df_players")
print(df_players.describe())
print("")
print("-------------------------------")
print("")
print("Información de df_ranking")
print(df_ranking.describe())
print("")
print("-------------------------------")
print("")
print("Información de df_teams")
print(df_teams.describe())

Información de df_games
            GAME_ID  HOME_TEAM_ID  VISITOR_TEAM_ID        SEASON  \
count  2.665100e+04  2.665100e+04     2.665100e+04  26651.000000   
mean   2.175487e+07  1.610613e+09     1.610613e+09   2012.113879   
std    5.570189e+06  8.638670e+00     8.659299e+00      5.587031   
min    1.030000e+07  1.610613e+09     1.610613e+09   2003.000000   
25%    2.070001e+07  1.610613e+09     1.610613e+09   2007.000000   
50%    2.120076e+07  1.610613e+09     1.610613e+09   2012.000000   
75%    2.180005e+07  1.610613e+09     1.610613e+09   2017.000000   
max    5.210021e+07  1.610613e+09     1.610613e+09   2022.000000   

       TEAM_ID_home      PTS_home   FG_PCT_home   FT_PCT_home  FG3_PCT_home  \
count  2.665100e+04  26552.000000  26552.000000  26552.000000  26552.000000   
mean   1.610613e+09    103.455898      0.460735      0.760377      0.356023   
std    8.638670e+00     13.283370      0.056676      0.100677      0.111164   
min    1.610613e+09     36.000000      0.250000

# Vamos a enfocarnos en los partidos entre 2005-2019

Del dataframe de games vamos a ver cuales partidos estan entre 2006-2019, cortamos la base hasta 2019 para evitar los datos en la pandemia de COVID-19

In [88]:
df_games["GAME_DATE_EST"] = pd.to_datetime(df_games["GAME_DATE_EST"])

filtro = (df_games["GAME_DATE_EST"].dt.year >= 2005) & (df_games["GAME_DATE_EST"].dt.year <= 2019)
df_games_fil = df_games.loc[filtro, ["GAME_DATE_EST", "GAME_ID", "TEAM_ID_home","TEAM_ID_away"]]

print(f"Partidos que aparecen en el dataframe de games: {df_games_fil['GAME_ID'].count():,}") 

df_games_unique = df_games_fil.drop_duplicates()

print(f"Partidos que aparecen en el dataframe de games sin repeticiones: {df_games_unique['GAME_ID'].count():,}") 

#df_games.info()

Partidos que aparecen en el dataframe de games: 20,939
Partidos que aparecen en el dataframe de games sin repeticiones: 20,939


Como ambos valores nos salieron igual, entonces tenemos 20,939 partidos entre 2005-2019. 
Vamos a ver cuantos partidos aparecen en la parte de juegos detalle. 

In [89]:
df_games_details_fil = df_games_details[df_games_details["GAME_ID"].isin(df_games_fil["GAME_ID"])]

print(f"Partidos que aparecen en el dataframe de games detail : {df_games_details_fil['GAME_ID'].nunique():,}") 

Partidos que aparecen en el dataframe de games detail : 20,939


Ahora vamos a revisar los jugadores que estan el tabla de games detail y los vamos a comparar con los que aparecen en el dataframe de players. \
Hasta ahora tenemos dos dataframes que nos van a ayudar a construir nuestra tablas de hechos y dimensiones:

-df_games_fil \
-df_games_details_fil 


In [90]:
print(df_games_details_fil["PLAYER_ID"].nunique())
print(df_players["PLAYER_ID"].nunique())


2215
1769


In [91]:
# Verificar si todos los PLAYER_ID de df_games_details_fil están en df_players
todos_contenidos = df_games_details_fil["PLAYER_ID"].isin(df_players["PLAYER_ID"]).all()
print("¿Todos los jugadores están contenidos en df_players?:", todos_contenidos)


¿Todos los jugadores están contenidos en df_players?: False


Como ya comprobamos que no todos los "PLAYER_ID" estan en df_players, vamos a hacer otro dataframe donde esten todos los jugadores


In [92]:
df_players_fil = df_games_details_fil[["PLAYER_ID","PLAYER_NAME"]].drop_duplicates()
df_players_fil.count()

PLAYER_ID      2215
PLAYER_NAME    2215
dtype: int64

Ahora vamos a ver el dataframe de teams,vamos a agregar una columna adicional con la conferencia a la que pertenence. 

In [93]:
conferencias = {
    # Conferencia Este
    "Hawks": "Eastern",
    "Celtics": "Eastern",
    "Nets": "Eastern",
    "Hornets": "Eastern",
    "Bulls": "Eastern",
    "Cavaliers": "Eastern",
    "Pistons": "Eastern",
    "Pacers": "Eastern",
    "Heat": "Eastern",
    "Bucks": "Eastern",
    "Knicks": "Eastern",
    "Magic": "Eastern",
    "76ers": "Eastern",
    "Raptors": "Eastern",
    "Wizards": "Eastern",
    # Conferencia Oeste
    "Mavericks": "Western",
    "Nuggets": "Western",
    "Warriors": "Western",
    "Rockets": "Western",
    "Clippers": "Western",
    "Lakers": "Western",
    "Grizzlies": "Western",
    "Timberwolves": "Western",
    "Pelicans": "Western",
    "Thunder": "Western",
    "Suns": "Western",
    "Trail Blazers": "Western",
    "Kings": "Western",
    "Spurs": "Western",
    "Jazz": "Western"
}

df_teams["CONFERENCE"] = df_teams["NICKNAME"].map(conferencias)

##Comprobacion para verificar que sean 15 equipos por conferencia
df_teams.groupby("CONFERENCE")["NICKNAME"].count()


CONFERENCE
Eastern    15
Western    15
Name: NICKNAME, dtype: int64

# Vamos a empezar con la construccion de nuestras tablas de hechos y dimensiones, tambien vamos a aplicar limpieza de los datos

In [94]:
#Funcion para pasar en minusculas los titulos de los dataframe y quitar espacios
def normalizar_columnas(df):
    df.columns = (
        df.columns
        .str.strip()          
        .str.lower()          
        .str.replace(" ", "_") 
    )
    return df

## Dim_team

In [95]:
df_teams.head()
dim_team = df_teams[["TEAM_ID","NICKNAME","CONFERENCE","CITY"]]
dim_team = normalizar_columnas(dim_team)
#dim_team.head(30)
dim_team.dtypes

team_id       int64
nickname        str
conference      str
city            str
dtype: object

## Dim_games

In [96]:
df_games_fil.head()

,GAME_DATE_EST,GAME_ID,TEAM_ID_home,TEAM_ID_away
3853,2019-12-31,21900497,1610612766,1610612738
3854,2019-12-31,21900498,1610612754,1610612755
3855,2019-12-31,21900499,1610612758,1610612746
3856,2019-12-31,21900500,1610612761,1610612739
3857,2019-12-31,21900501,1610612745,1610612743


In [97]:
dim_team.head()

,team_id,nickname,conference,city
0,1610612737,Hawks,Eastern,Atlanta
1,1610612738,Celtics,Eastern,Boston
2,1610612740,Pelicans,Western,New Orleans
3,1610612741,Bulls,Eastern,Chicago
4,1610612742,Mavericks,Western,Dallas


Hacer el cruce con el id del equipo local

In [98]:
df_games_fil = df_games_fil.merge(
    dim_team[['team_id', 'nickname']],
    left_on='TEAM_ID_home',
    right_on='team_id',
    how='left'
)

df_games_fil = df_games_fil.rename(columns={"team_id":"team_id_home","nickname": "nickname_home"})
df_games_fil.head()


,GAME_DATE_EST,GAME_ID,TEAM_ID_home,TEAM_ID_away,team_id_home,nickname_home
0,2019-12-31,21900497,1610612766,1610612738,1610612766,Hornets
1,2019-12-31,21900498,1610612754,1610612755,1610612754,Pacers
2,2019-12-31,21900499,1610612758,1610612746,1610612758,Kings
3,2019-12-31,21900500,1610612761,1610612739,1610612761,Raptors
4,2019-12-31,21900501,1610612745,1610612743,1610612745,Rockets


Hacer el cruce con el equipo visitante

In [99]:
df_games_fil = df_games_fil.merge(
    dim_team[['team_id', 'nickname']],
    left_on='TEAM_ID_away',
    right_on='team_id',
    how='left'
)

df_games_fil = df_games_fil.rename(columns={"team_id":"team_id_away","nickname": "nickname_away"})
df_games_fil.head()

,GAME_DATE_EST,GAME_ID,TEAM_ID_home,TEAM_ID_away,team_id_home,nickname_home,team_id_away,nickname_away
0,2019-12-31,21900497,1610612766,1610612738,1610612766,Hornets,1610612738,Celtics
1,2019-12-31,21900498,1610612754,1610612755,1610612754,Pacers,1610612755,76ers
2,2019-12-31,21900499,1610612758,1610612746,1610612758,Kings,1610612746,Clippers
3,2019-12-31,21900500,1610612761,1610612739,1610612761,Raptors,1610612739,Cavaliers
4,2019-12-31,21900501,1610612745,1610612743,1610612745,Rockets,1610612743,Nuggets


In [100]:
df_games_fil.head()

,GAME_DATE_EST,GAME_ID,TEAM_ID_home,TEAM_ID_away,team_id_home,nickname_home,team_id_away,nickname_away
0,2019-12-31,21900497,1610612766,1610612738,1610612766,Hornets,1610612738,Celtics
1,2019-12-31,21900498,1610612754,1610612755,1610612754,Pacers,1610612755,76ers
2,2019-12-31,21900499,1610612758,1610612746,1610612758,Kings,1610612746,Clippers
3,2019-12-31,21900500,1610612761,1610612739,1610612761,Raptors,1610612739,Cavaliers
4,2019-12-31,21900501,1610612745,1610612743,1610612745,Rockets,1610612743,Nuggets


In [101]:
df_games_fil_2 = df_games_fil[["GAME_ID","nickname_home","nickname_away","GAME_DATE_EST"]]
df_games_fil_2["game_name"] = df_games_fil_2["nickname_home"] + " vs " + df_games_fil_2["nickname_away"] + " on " + df_games_fil_2["GAME_DATE_EST"].astype(str)
df_games_fil_2.head()

,GAME_ID,nickname_home,nickname_away,GAME_DATE_EST,game_name
0,21900497,Hornets,Celtics,2019-12-31,Hornets vs Celtics on 2019-12-31
1,21900498,Pacers,76ers,2019-12-31,Pacers vs 76ers on 2019-12-31
2,21900499,Kings,Clippers,2019-12-31,Kings vs Clippers on 2019-12-31
3,21900500,Raptors,Cavaliers,2019-12-31,Raptors vs Cavaliers on 2019-12-31
4,21900501,Rockets,Nuggets,2019-12-31,Rockets vs Nuggets on 2019-12-31


In [102]:
dim_games = df_games_fil_2[["GAME_ID","game_name"]]
dim_games = normalizar_columnas(dim_games)
dim_games.dtypes

game_id      int64
game_name      str
dtype: object

## Dim_player

In [103]:
#df_players_fil.head()
dim_player = df_players_fil.copy()
dim_player = normalizar_columnas(dim_player)
dim_player.dtypes



player_id      int64
player_name      str
dtype: object

## Dim_star_position

In [104]:
df_games_details_fil_2 = df_games_details_fil.copy()
posiciones_map = {
    "G": 1,
    "F": 2,
    "C": 3
}
df_games_details_fil_2["star_position_id"] = (
    df_games_details_fil_2["START_POSITION"]
    .map(posiciones_map)
    .fillna(4)
)

df_games_details_fil_2["star_position_id"] = df_games_details_fil_2["star_position_id"].astype(int)

df_games_details_fil_2.head()


#df_games_details_fil_2.head()

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,star_position_id
101771,21900497,1610612738,BOS,Boston,202330,Gordon Hayward,NaN,F,NaN,34:56,...,5.0,10.0,6.0,0.0,0.0,2.0,0.0,21.0,12.0,2
101772,21900497,1610612738,BOS,Boston,1628369,Jayson Tatum,NaN,F,NaN,36:15,...,5.0,7.0,1.0,3.0,1.0,3.0,0.0,24.0,15.0,2
101773,21900497,1610612738,BOS,Boston,1628464,Daniel Theis,NaN,C,NaN,22:31,...,4.0,6.0,0.0,2.0,1.0,0.0,2.0,5.0,2.0,3
101774,21900497,1610612738,BOS,Boston,203935,Marcus Smart,NaN,G,NaN,30:29,...,4.0,5.0,7.0,1.0,0.0,1.0,4.0,7.0,9.0,1
101775,21900497,1610612738,BOS,Boston,202689,Kemba Walker,NaN,G,NaN,30:40,...,2.0,4.0,7.0,3.0,1.0,1.0,0.0,22.0,5.0,1


Vamos a crear la dimension star position

In [105]:
data = [
    {"start_position_id": 1, "position_name": "Guard",   "flag_holder": 1},
    {"start_position_id": 2, "position_name": "Forward", "flag_holder": 1},
    {"start_position_id": 3, "position_name": "Center",  "flag_holder": 1},
    {"start_position_id": 4, "position_name": "Bench",  "flag_holder": 0}
]

dim_star_position = pd.DataFrame(data)

#dim_star_position.head()
dim_star_position.dtypes


start_position_id    int64
position_name          str
flag_holder          int64
dtype: object

## Dim date

In [106]:
df_games_details_fil_2.head()

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,star_position_id
101771,21900497,1610612738,BOS,Boston,202330,Gordon Hayward,NaN,F,NaN,34:56,...,5.0,10.0,6.0,0.0,0.0,2.0,0.0,21.0,12.0,2
101772,21900497,1610612738,BOS,Boston,1628369,Jayson Tatum,NaN,F,NaN,36:15,...,5.0,7.0,1.0,3.0,1.0,3.0,0.0,24.0,15.0,2
101773,21900497,1610612738,BOS,Boston,1628464,Daniel Theis,NaN,C,NaN,22:31,...,4.0,6.0,0.0,2.0,1.0,0.0,2.0,5.0,2.0,3
101774,21900497,1610612738,BOS,Boston,203935,Marcus Smart,NaN,G,NaN,30:29,...,4.0,5.0,7.0,1.0,0.0,1.0,4.0,7.0,9.0,1
101775,21900497,1610612738,BOS,Boston,202689,Kemba Walker,NaN,G,NaN,30:40,...,2.0,4.0,7.0,3.0,1.0,1.0,0.0,22.0,5.0,1


In [107]:
df_games_aux = df_games_fil[["GAME_ID", "GAME_DATE_EST"]]

df_games_details_fil_2 = df_games_details_fil_2.merge(df_games_aux, 
                          left_on="GAME_ID", 
                          right_on="GAME_ID", 
                          how="left")

df_games_details_fil_2.head() 

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,star_position_id,GAME_DATE_EST
0,21900497,1610612738,BOS,Boston,202330,Gordon Hayward,NaN,F,NaN,34:56,...,10.0,6.0,0.0,0.0,2.0,0.0,21.0,12.0,2,2019-12-31
1,21900497,1610612738,BOS,Boston,1628369,Jayson Tatum,NaN,F,NaN,36:15,...,7.0,1.0,3.0,1.0,3.0,0.0,24.0,15.0,2,2019-12-31
2,21900497,1610612738,BOS,Boston,1628464,Daniel Theis,NaN,C,NaN,22:31,...,6.0,0.0,2.0,1.0,0.0,2.0,5.0,2.0,3,2019-12-31
3,21900497,1610612738,BOS,Boston,203935,Marcus Smart,NaN,G,NaN,30:29,...,5.0,7.0,1.0,0.0,1.0,4.0,7.0,9.0,1,2019-12-31
4,21900497,1610612738,BOS,Boston,202689,Kemba Walker,NaN,G,NaN,30:40,...,4.0,7.0,3.0,1.0,1.0,0.0,22.0,5.0,1,2019-12-31


In [108]:
fecha_min = df_games_details_fil_2["GAME_DATE_EST"].min()
fecha_max = df_games_details_fil_2["GAME_DATE_EST"].max()

print("Fecha mínima:", fecha_min)
print("Fecha máxima:", fecha_max)

#df_games_details_fil_2.dtypes

df_games_details_fil_2.dtypes

Fecha mínima: 2005-01-01 00:00:00
Fecha máxima: 2019-12-31 00:00:00


GAME_ID                       int64
TEAM_ID                       int64
TEAM_ABBREVIATION               str
TEAM_CITY                       str
PLAYER_ID                     int64
PLAYER_NAME                     str
NICKNAME                        str
START_POSITION                  str
COMMENT                         str
MIN                             str
FGM                         float64
FGA                         float64
FG_PCT                      float64
FG3M                        float64
FG3A                        float64
FG3_PCT                     float64
FTM                         float64
FTA                         float64
FT_PCT                      float64
OREB                        float64
DREB                        float64
REB                         float64
AST                         float64
STL                         float64
BLK                         float64
TO                          float64
PF                          float64
PTS                         

In [109]:
# 1. Rango denso de fechas (una por día)
fechas = pd.date_range(start="2005-01-01", end="2019-12-31", freq="D")
dim_date = pd.DataFrame({"full_date": fechas})

# 2. Smart key YYYYMMDD como entero (la PK no es surrogate, es derivada)
dim_date["date_id"] = (
    dim_date["full_date"].dt.year * 10000
    + dim_date["full_date"].dt.month * 100
    + dim_date["full_date"].dt.day
)

# 3. Atributos derivados — todos vienen del accesor .dt
dim_date["year"]               = dim_date["full_date"].dt.year
dim_date["quarter"]            = dim_date["full_date"].dt.quarter
dim_date["month_number"]       = dim_date["full_date"].dt.month
dim_date["week_of_year"]       = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["day_of_month"]       = dim_date["full_date"].dt.day
dim_date["day_of_week_number"] = dim_date["full_date"].dt.dayofweek + 1  # ISO: lunes=1
dim_date["is_weekend"]         = dim_date["day_of_week_number"].isin([6, 7])

# Nombres en español sin depender del locale del sistema
meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
         "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
dias  = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

dim_date["month_name"]       = dim_date["month_number"].map(lambda m: meses[m-1])
dim_date["day_of_week_name"] = dim_date["day_of_week_number"].map(lambda d: dias[d-1])

# Reordenar columnas para que coincidan con el DDL
dim_date = dim_date[[
    "date_id", "full_date", "year", "quarter",
    "month_number", "month_name", "week_of_year",
    "day_of_month", "day_of_week_number", "day_of_week_name",
    "is_weekend"
]]

print(f"dim_date: {dim_date.shape}")
dim_date.head()

dim_date: (5478, 11)


,date_id,full_date,year,quarter,month_number,month_name,week_of_year,day_of_month,day_of_week_number,day_of_week_name,is_weekend
0,20050101,2005-01-01,2005,1,1,Enero,53,1,6,Sábado,True
1,20050102,2005-01-02,2005,1,1,Enero,53,2,7,Domingo,True
2,20050103,2005-01-03,2005,1,1,Enero,1,3,1,Lunes,False
3,20050104,2005-01-04,2005,1,1,Enero,1,4,2,Martes,False
4,20050105,2005-01-05,2005,1,1,Enero,1,5,3,Miércoles,False


## Tabla de hechos

In [110]:
df_games_details_fil_2.head(10)

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,star_position_id,GAME_DATE_EST
0,21900497,1610612738,BOS,Boston,202330,Gordon Hayward,NaN,F,NaN,34:56,...,10.0,6.0,0.0,0.0,2.0,0.0,21.0,12.0,2,2019-12-31
1,21900497,1610612738,BOS,Boston,1628369,Jayson Tatum,NaN,F,NaN,36:15,...,7.0,1.0,3.0,1.0,3.0,0.0,24.0,15.0,2,2019-12-31
2,21900497,1610612738,BOS,Boston,1628464,Daniel Theis,NaN,C,NaN,22:31,...,6.0,0.0,2.0,1.0,0.0,2.0,5.0,2.0,3,2019-12-31
3,21900497,1610612738,BOS,Boston,203935,Marcus Smart,NaN,G,NaN,30:29,...,5.0,7.0,1.0,0.0,1.0,4.0,7.0,9.0,1,2019-12-31
4,21900497,1610612738,BOS,Boston,202689,Kemba Walker,NaN,G,NaN,30:40,...,4.0,7.0,3.0,1.0,1.0,0.0,22.0,5.0,1,2019-12-31
5,21900497,1610612738,BOS,Boston,1629750,Javonte Green,NaN,NaN,NaN,9:15,...,3.0,0.0,1.0,0.0,1.0,0.0,2.0,4.0,4,2019-12-31
6,21900497,1610612738,BOS,Boston,1629684,Grant Williams,NaN,NaN,NaN,9:10,...,2.0,0.0,0.0,0.0,0.0,2.0,0.0,3.0,4,2019-12-31
7,21900497,1610612738,BOS,Boston,202683,Enes Kanter,NaN,NaN,NaN,22:35,...,14.0,2.0,0.0,6.0,0.0,5.0,13.0,15.0,4,2019-12-31
8,21900497,1610612738,BOS,Boston,202954,Brad Wanamaker,NaN,NaN,NaN,21:09,...,2.0,1.0,1.0,0.0,2.0,2.0,7.0,13.0,4,2019-12-31
9,21900497,1610612738,BOS,Boston,1628400,Semi Ojeleye,NaN,NaN,NaN,20:40,...,1.0,1.0,1.0,0.0,0.0,1.0,6.0,8.0,4,2019-12-31


In [111]:
df_games_details_fil_2["date_id"] = (
    df_games_details_fil_2["GAME_DATE_EST"].dt.year * 10000
    + df_games_details_fil_2["GAME_DATE_EST"].dt.month * 100
    + df_games_details_fil_2["GAME_DATE_EST"].dt.day
)

df_games_details_fil_2.head()

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,star_position_id,GAME_DATE_EST,date_id
0,21900497,1610612738,BOS,Boston,202330,Gordon Hayward,NaN,F,NaN,34:56,...,6.0,0.0,0.0,2.0,0.0,21.0,12.0,2,2019-12-31,20191231
1,21900497,1610612738,BOS,Boston,1628369,Jayson Tatum,NaN,F,NaN,36:15,...,1.0,3.0,1.0,3.0,0.0,24.0,15.0,2,2019-12-31,20191231
2,21900497,1610612738,BOS,Boston,1628464,Daniel Theis,NaN,C,NaN,22:31,...,0.0,2.0,1.0,0.0,2.0,5.0,2.0,3,2019-12-31,20191231
3,21900497,1610612738,BOS,Boston,203935,Marcus Smart,NaN,G,NaN,30:29,...,7.0,1.0,0.0,1.0,4.0,7.0,9.0,1,2019-12-31,20191231
4,21900497,1610612738,BOS,Boston,202689,Kemba Walker,NaN,G,NaN,30:40,...,7.0,3.0,1.0,1.0,0.0,22.0,5.0,1,2019-12-31,20191231


Vamos a eleminar los registros donde los jugadores no hayan jugado ningun segundo. Es decir, donde la columna de MIN sea NaN

In [112]:
# Eliminar filas donde MIN sea nulo
df_games_details_fil_2 = df_games_details_fil_2.dropna(subset=["MIN"])
df_games_details_fil_2["MIN"].isnull().sum()

np.int64(0)

Ahora vamos a trabjar con la tabla de "MIN" la vamos a pasar a segundos jugados. 

In [113]:
#df_games_details_fil_2.head()

# Separar por el carácter ":"
df_games_details_fil_2[["hora_min", "hora_seg"]] = df_games_details_fil_2["MIN"].str.split(":", expand=True)

df_games_details_fil_2["hora_min"] = df_games_details_fil_2["hora_min"].fillna(0).astype(int)
df_games_details_fil_2["hora_seg"] = df_games_details_fil_2["hora_seg"].fillna(0).astype(int)


df_games_details_fil_2.head()



,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,BLK,TO,PF,PTS,PLUS_MINUS,star_position_id,GAME_DATE_EST,date_id,hora_min,hora_seg
0,21900497,1610612738,BOS,Boston,202330,Gordon Hayward,NaN,F,NaN,34:56,...,0.0,2.0,0.0,21.0,12.0,2,2019-12-31,20191231,34,56
1,21900497,1610612738,BOS,Boston,1628369,Jayson Tatum,NaN,F,NaN,36:15,...,1.0,3.0,0.0,24.0,15.0,2,2019-12-31,20191231,36,15
2,21900497,1610612738,BOS,Boston,1628464,Daniel Theis,NaN,C,NaN,22:31,...,1.0,0.0,2.0,5.0,2.0,3,2019-12-31,20191231,22,31
3,21900497,1610612738,BOS,Boston,203935,Marcus Smart,NaN,G,NaN,30:29,...,0.0,1.0,4.0,7.0,9.0,1,2019-12-31,20191231,30,29
4,21900497,1610612738,BOS,Boston,202689,Kemba Walker,NaN,G,NaN,30:40,...,1.0,1.0,0.0,22.0,5.0,1,2019-12-31,20191231,30,40


In [114]:
df_games_details_fil_2["sec"] = df_games_details_fil_2["hora_min"]*60 + df_games_details_fil_2["hora_seg"]

df_games_details_fil_2.head()

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,TO,PF,PTS,PLUS_MINUS,star_position_id,GAME_DATE_EST,date_id,hora_min,hora_seg,sec
0,21900497,1610612738,BOS,Boston,202330,Gordon Hayward,NaN,F,NaN,34:56,...,2.0,0.0,21.0,12.0,2,2019-12-31,20191231,34,56,2096
1,21900497,1610612738,BOS,Boston,1628369,Jayson Tatum,NaN,F,NaN,36:15,...,3.0,0.0,24.0,15.0,2,2019-12-31,20191231,36,15,2175
2,21900497,1610612738,BOS,Boston,1628464,Daniel Theis,NaN,C,NaN,22:31,...,0.0,2.0,5.0,2.0,3,2019-12-31,20191231,22,31,1351
3,21900497,1610612738,BOS,Boston,203935,Marcus Smart,NaN,G,NaN,30:29,...,1.0,4.0,7.0,9.0,1,2019-12-31,20191231,30,29,1829
4,21900497,1610612738,BOS,Boston,202689,Kemba Walker,NaN,G,NaN,30:40,...,1.0,0.0,22.0,5.0,1,2019-12-31,20191231,30,40,1840


In [115]:
df_games_details_fil_3 = df_games_details_fil_2[["GAME_ID","TEAM_ID","date_id", "PLAYER_ID", "star_position_id", "sec","FGM","FGA","FG3M","FG3A","FTM","FTA","REB","AST","STL","BLK","TO","PF","PTS"]]

columnas = ["FGM","FGA","FG3M","FG3A","FTM","FTA",
              "REB","AST","STL","BLK","TO","PF","PTS"]

df_games_details_fil_3[columnas] = df_games_details_fil_3[columnas].astype(int)

df_games_details_fil_3["PTS"] = df_games_details_fil_3["FGM"]*2 + df_games_details_fil_3["FG3M"] *3 + df_games_details_fil_3["FTM"] 

df_games_details_fil_3.head()

,GAME_ID,TEAM_ID,date_id,PLAYER_ID,star_position_id,sec,FGM,FGA,FG3M,FG3A,FTM,FTA,REB,AST,STL,BLK,TO,PF,PTS
0,21900497,1610612738,20191231,202330,2,2096,9,14,3,6,0,0,10,6,0,0,2,0,27
1,21900497,1610612738,20191231,1628369,2,2175,10,18,4,7,0,0,7,1,3,1,3,0,32
2,21900497,1610612738,20191231,1628464,3,1351,2,9,0,2,1,2,6,0,2,1,0,2,5
3,21900497,1610612738,20191231,203935,1,1829,3,12,1,7,0,1,5,7,1,0,1,4,9
4,21900497,1610612738,20191231,202689,1,1840,9,19,3,7,1,1,4,7,3,1,1,0,28


In [116]:
fact_statistical = df_games_details_fil_3.copy()
fact_statistical =  normalizar_columnas(fact_statistical)

fact_statistical.dtypes


game_id             int64
team_id             int64
date_id             int32
player_id           int64
star_position_id    int64
sec                 int64
fgm                 int64
fga                 int64
fg3m                int64
fg3a                int64
ftm                 int64
fta                 int64
reb                 int64
ast                 int64
stl                 int64
blk                 int64
to                  int64
pf                  int64
pts                 int64
dtype: object

# Vamos a verificar nuestra tabla de hechos y dimensiones 

Antes de hacer la carga vamos a cambiar la columna to de la tabla de hechos porque para SQL es una palabra reservada

In [117]:
fact_statistical.head()

,game_id,team_id,date_id,player_id,star_position_id,sec,fgm,fga,fg3m,fg3a,ftm,fta,reb,ast,stl,blk,to,pf,pts
0,21900497,1610612738,20191231,202330,2,2096,9,14,3,6,0,0,10,6,0,0,2,0,27
1,21900497,1610612738,20191231,1628369,2,2175,10,18,4,7,0,0,7,1,3,1,3,0,32
2,21900497,1610612738,20191231,1628464,3,1351,2,9,0,2,1,2,6,0,2,1,0,2,5
3,21900497,1610612738,20191231,203935,1,1829,3,12,1,7,0,1,5,7,1,0,1,4,9
4,21900497,1610612738,20191231,202689,1,1840,9,19,3,7,1,1,4,7,3,1,1,0,28


In [118]:
fact_statistical = fact_statistical.rename(columns={"to": "tos","star_position_id":"start_position_id"})

In [119]:
datasets = {
    "dim_team": dim_team,
    "dim_games": dim_games,
    "dim_player": dim_player,
    "dim_star_position": dim_star_position,
    "dim_date": dim_date,
    "fact_statistical": fact_statistical
}

for name, df in datasets.items():
    print(f"\n📂 {name}")
    print(df.dtypes)


📂 dim_team
team_id       int64
nickname        str
conference      str
city            str
dtype: object

📂 dim_games
game_id      int64
game_name      str
dtype: object

📂 dim_player
player_id      int64
player_name      str
dtype: object

📂 dim_star_position
start_position_id    int64
position_name          str
flag_holder          int64
dtype: object

📂 dim_date
date_id                        int32
full_date             datetime64[us]
year                           int32
quarter                        int32
month_number                   int32
month_name                       str
week_of_year                   int64
day_of_month                   int32
day_of_week_number             int32
day_of_week_name                 str
is_weekend                      bool
dtype: object

📂 fact_statistical
game_id              int64
team_id              int64
date_id              int32
player_id            int64
start_position_id    int64
sec                  int64
fgm                  int64
f

In [120]:
for name, df in datasets.items():
    print(f"\n📂 {name}")
    print(df.describe(include="all"))


📂 dim_team
             team_id nickname conference         city
count   3.000000e+01       30         30           30
unique           NaN       30          2           29
top              NaN    Hawks    Eastern  Los Angeles
freq             NaN        1         15            2
mean    1.610613e+09      NaN        NaN          NaN
std     8.803408e+00      NaN        NaN          NaN
min     1.610613e+09      NaN        NaN          NaN
25%     1.610613e+09      NaN        NaN          NaN
50%     1.610613e+09      NaN        NaN          NaN
75%     1.610613e+09      NaN        NaN          NaN
max     1.610613e+09      NaN        NaN          NaN

📂 dim_games
             game_id                         game_name
count   2.093900e+04                             20939
unique           NaN                             20939
top              NaN  Hornets vs Celtics on 2019-12-31
freq             NaN                                 1
mean    2.166651e+07                               N

In [121]:
for name, df in datasets.items():
    print(f"\n📂 {name}")
    print(df.shape[0])


📂 dim_team
30

📂 dim_games
20939

📂 dim_player
2215

📂 dim_star_position
4

📂 dim_date
5478

📂 fact_statistical
440373


## Vamos a empezar con la carga

In [122]:
import pandas as pd
from sqlalchemy import create_engine, text

In [123]:
AURORA_HOST     = "aurora-mod4.cluster-cvzppmurq22n.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "CapI3007*"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

In [124]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS nba_dwh_py"))

print("✓ schema nba_dwh_py listo")

✓ schema nba_dwh_py listo


In [125]:
# Carga ordenada de las 5 dims + fact_sales
tablas_en_orden = [
    ("dim_team", dim_team),
    ("dim_games", dim_games),
    ("dim_player", dim_player),
    ("dim_start_position", dim_star_position),
    ("dim_date", dim_date),
    ("fact_statistical", fact_statistical)
]

for nombre, df in tablas_en_orden:
    df.to_sql(
        name=nombre,
        con=engine,
        schema="nba_dwh_py",
        if_exists="append",
        index=False,
        chunksize=1000,
        method="multi",
    )
    print(f"✓ {nombre:14s} {len(df):>5} filas")

print("\nCarga completa.")

✓ dim_team          30 filas
✓ dim_games      20939 filas
✓ dim_player      2215 filas
✓ dim_start_position     4 filas
✓ dim_date        5478 filas
✓ fact_statistical 440373 filas

Carga completa.


Corroboramos que los totales sean iguales. 

In [126]:
datasets = {
    "dim_team": dim_team,
    "dim_games": dim_games,
    "dim_player": dim_player,
    "dim_star_position": dim_star_position,
    "dim_date": dim_date,
    "fact_statistical": fact_statistical
}

for name, df in datasets.items():
    print(f"\n📂 {name}")
    print(df.shape[0])


📂 dim_team
30

📂 dim_games
20939

📂 dim_player
2215

📂 dim_star_position
4

📂 dim_date
5478

📂 fact_statistical
440373
